In [ ]:
import pandas as pd
import networkx as nx

In [ ]:
data = pd.read_csv('../../data/data/processed/fdemand.csv')

In [ ]:
data_no3000 = data[data['end_station'] != 3000]

In [ ]:
data_no3000.head()

### Making the Data Set

In [ ]:
# Define all_station_coords_df_full
# Extract unique station coordinates from the main data
# We'll combine start and end station information to get a comprehensive list.

# Collect start station info (without 'start_name' as it's missing)
start_stations = data[['start_station', 'st_lat', 'st_lon']].drop_duplicates().copy()
start_stations.rename(columns={'start_station': 'station_id',
                               'st_lat': 'latitude',
                               'st_lon': 'longitude'}, inplace=True)

In [ ]:
# Add a placeholder for station_name for start_stations
start_stations['station_name'] = None

In [ ]:
# Collect end station info (with 'end_name')
end_stations = data[['end_station', 'end_lat', 'end_lon', 'end_name']].drop_duplicates().copy()
end_stations.rename(columns={'end_station': 'station_id',
                             'end_lat': 'latitude',
                             'end_lon': 'longitude',
                             'end_name': 'station_name'}, inplace=True)

In [ ]:
# Combine and remove duplicates to get all unique station coordinates and names
# Use a common set of columns for concat to ensure they align
all_station_coords_df_full = pd.concat([start_stations[['station_id', 'latitude', 'longitude', 'station_name']],
                                        end_stations[['station_id', 'latitude', 'longitude', 'station_name']]]) \
    .drop_duplicates(subset=['station_id']).reset_index(drop=True)

In [ ]:
# Ensure station_id is integer type for consistent merging
all_station_coords_df_full['station_id'] = all_station_coords_df_full['station_id'].astype(int)

In [ ]:
# Preprocessing: Calculate daily net change per station from data_no3000
# Ensure 'st_year', 'st_month', 'st_day' are used to create a date column
# This assumes data_no3000 contains 'start_station', 'end_station', 'st_year', 'st_month', 'st_day'

# Create a daily identifier
# Make a copy to avoid SettingWithCopyWarning
data_for_net_change = data_no3000.copy()
data_for_net_change['date'] = pd.to_datetime(
    {'year': data_for_net_change['st_year'], 'month': data_for_net_change['st_month'],
     'day': data_for_net_change['st_day']})

In [ ]:
# Count bikes leaving each station per day
departures = data_for_net_change.groupby(['date', 'start_station']).size().reset_index(name='departures')
departures.rename(columns={'start_station': 'station_id'}, inplace=True)

In [ ]:
# Count bikes arriving at each station per day
arrivals = data_for_net_change.groupby(['date', 'end_station']).size().reset_index(name='arrivals')
arrivals.rename(columns={'end_station': 'station_id'}, inplace=True)

In [ ]:
# Merge departures and arrivals
daily_station_activity = pd.merge(departures, arrivals, on=['date', 'station_id'], how='outer').fillna(0)


In [ ]:
# Calculate net change
daily_station_activity['net_change'] = daily_station_activity['arrivals'] - daily_station_activity['departures']

In [ ]:
# 1. Create a directed graph representing the bike-sharing network
G = nx.DiGraph()

In [ ]:
# Add all unique stations as nodes
all_unique_stations = pd.concat([data_no3000['start_station'], data_no3000['end_station']]).unique()
G.add_nodes_from(all_unique_stations)

In [ ]:
# Add edges with weights (number of trips) between start and end stations
trip_counts = data_no3000.groupby(['start_station', 'end_station']).size().reset_index(name='trip_count')
for _, row in trip_counts.iterrows():
    G.add_edge(row['start_station'], row['end_station'], weight=row['trip_count'])

In [ ]:
# 2. Calculate Centrality Measures
centrality_measures = {
    'station_id': [],
    'degree_centrality_in': [],
    'degree_centrality_out': [],
    'betweenness_centrality_unweighted': [],
    'closeness_centrality_unweighted': [],
    'eigenvector_centrality_weighted': []
}

In [ ]:
in_degree_centrality = nx.in_degree_centrality(G)
out_degree_centrality = nx.out_degree_centrality(G)

In [ ]:
# Use unweighted centrality for betweenness and closeness for a standard interpretation of 'paths'
betweenness_unweighted = nx.betweenness_centrality(G)
closeness_unweighted = nx.closeness_centrality(G)

In [ ]:
# Use weighted eigenvector centrality, where higher trip count means more influence
try:
    eigenvector_weighted = nx.eigenvector_centrality(G, weight='weight', max_iter=1000, tol=1e-06)
except nx.PowerIterationFailedConvergence:
    print("Eigenvector centrality did not converge for some nodes. Assigning 0 to non-converged nodes.")
    eigenvector_weighted = {node: 0 for node in G.nodes()}
except Exception as e:
    print(f"Error calculating eigenvector centrality: {e}. Assigning 0 to all nodes.")
    eigenvector_weighted = {node: 0 for node in G.nodes()}

In [ ]:
for station_id in G.nodes():
    centrality_measures['station_id'].append(station_id)
    centrality_measures['degree_centrality_in'].append(in_degree_centrality.get(station_id, 0))
    centrality_measures['degree_centrality_out'].append(out_degree_centrality.get(station_id, 0))
    centrality_measures['betweenness_centrality_unweighted'].append(betweenness_unweighted.get(station_id, 0))
    centrality_measures['closeness_centrality_unweighted'].append(closeness_unweighted.get(station_id, 0))
    centrality_measures['eigenvector_centrality_weighted'].append(eigenvector_weighted.get(station_id, 0))


In [ ]:
station_centrality_df = pd.DataFrame(centrality_measures)

In [ ]:
# 3. Prepare base DataFrame for modeling: `daily_station_activity`
# This DataFrame contains 'date', 'station_id', 'departures', 'arrivals', 'net_change'
modeling_df = daily_station_activity.copy()

In [ ]:
# 4. Add Time-based Features
modeling_df['weekday'] = modeling_df['date'].dt.weekday  # Monday=0, Sunday=6
modeling_df['dayofyear'] = modeling_df['date'].dt.dayofyear
modeling_df['month'] = modeling_df['date'].dt.month
modeling_df['is_weekend'] = modeling_df['date'].dt.weekday.isin([5, 6])

In [ ]:
# 5. Aggregate Daily Weather Data
# Create a 'date' column in data_no3000 to merge with modeling_df
weather_data_temp = data_no3000.copy()
weather_data_temp['date'] = pd.to_datetime(
    {'year': weather_data_temp['st_year'], 'month': weather_data_temp['st_month'], 'day': weather_data_temp['st_day']})

weather_features = weather_data_temp.groupby('date')[
    ['avg_wind', 'precip', 'snow', 'snow_ground', 'max_temp', 'min_temp']].mean().reset_index()

In [ ]:
# 6. Merge all features into the modeling_df
# Merge centrality measures
modeling_df = pd.merge(modeling_df, station_centrality_df, on='station_id', how='left')

# Merge weather features
modeling_df = pd.merge(modeling_df, weather_features, on='date', how='left')

In [ ]:
# Handle potential NaNs after merging
# Fill centrality NaNs with 0 (e.g., if a station existed in daily_station_activity but not in the graph)
centrality_cols = [col for col in station_centrality_df.columns if col != 'station_id']
for col in centrality_cols:
    modeling_df[col] = modeling_df[col].fillna(0)

In [ ]:
# Fill weather NaNs, assuming weather features are generally available. Forward fill, then fill remaining with 0 or mean.
modeling_df.sort_values(by=['date', 'station_id'], inplace=True)
modeling_df[weather_features.columns.drop('date')] = modeling_df[weather_features.columns.drop('date')].ffill()

In [ ]:
# Fill any remaining NaNs at the beginning of the series or if ffill couldn't propagate
modeling_df[weather_features.columns.drop('date')] = modeling_df[weather_features.columns.drop('date')].fillna(0)

In [ ]:
# Display the head and info of the prepared dataset
print("Prepared dataset for XGBoost model (first 5 rows):")
display(modeling_df.head())

In [ ]:
print("\nInformation about the prepared dataset:")
modeling_df.info()

In [ ]:
# save modeling_df to modeling_day.csv for modeling days.
modeling_df.to_csv('modeling_day.csv', index=False)